In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

# Project root
# Find the folder Python is currently working in, then go up one folder
PROJECT_ROOT = Path.cwd().parent

# Data directories
RAW_DIR = PROJECT_ROOT / "data" / "raw"

TRIP_FILE = RAW_DIR / "yellow_tripdata_2026-01.parquet"
ZONE_FILE = RAW_DIR / "taxi_zone_lookup.csv"

RAW_DIR = PROJECT_ROOT / "data" / "raw"
BRONZE_DIR = PROJECT_ROOT / "data" / "bronze"
SILVER_DIR = PROJECT_ROOT / "data" / "silver"
GOLD_DIR = PROJECT_ROOT / "data" / "gold"

print("Trip file exists:", TRIP_FILE.exists())
print("Zone file exists:", ZONE_FILE.exists())


Trip file exists: True
Zone file exists: True


In [29]:
# 1. Inspect Parquet file metadata and storage schema
parquet_file = pq.ParquetFile(TRIP_FILE)

print("\n=== Parquet File Information ===")
print(f"File size: {TRIP_FILE.stat().st_size / 1024**2:.2f} MB")
print(f"Number of rows: {parquet_file.metadata.num_rows:,}")
print(f"Row groups: {parquet_file.metadata.num_row_groups}")
print("\nStorage schema:")
print(parquet_file.schema_arrow)

# 2. Load trips and inspect columns, data types, and missing values
trips = pd.read_parquet(TRIP_FILE)

summary = pd.DataFrame({
    "dtype": trips.dtypes.astype(str),
    "missing_count": trips.isna().sum(),
    "missing_pct": trips.isna().mean().mul(100).round(2),
})

print("\n=== Trip Columns and Missing Values ===")
print(summary.to_string())

print("\n=== First 5 Rows ===")
print(trips.head().to_string(index=False))

# 3. Inspect datetime ranges; create temporary variables only and do not overwrite original columns
month_start = pd.Timestamp("2026-01-01")
month_end = pd.Timestamp("2026-02-01")

for column in ["tpep_pickup_datetime", "tpep_dropoff_datetime"]:
    print(f"\n=== {column} ===")

    if column not in trips.columns:
        print("Column is missing")
        continue

    timestamps = pd.to_datetime(trips[column], errors="coerce")
    invalid_count = (trips[column].notna() & timestamps.isna()).sum()
    outside_month = (
        timestamps.notna()
        & ((timestamps < month_start) | (timestamps >= month_end))
    )

    print(f"Earliest timestamp: {timestamps.min()}")
    print(f"Latest timestamp: {timestamps.max()}")
    print(f"Missing or unparseable: {timestamps.isna().sum():,}")
    print(f"Non-null but unparseable: {invalid_count:,}")
    print(f"Outside January 2026: {outside_month.sum():,}")

    if column == "tpep_pickup_datetime":
        print("\nRow count by pickup date:")
        print(
            timestamps.dt.date.value_counts()
            .sort_index()
            .to_string()
        )

# 4. Inspect taxi zone lookup
zones = pd.read_csv(ZONE_FILE)

print("\n=== Taxi Zone Lookup ===")
print(f"Number of rows: {len(zones):,}")

zone_summary = pd.DataFrame({
    "dtype": zones.dtypes.astype(str),
    "missing_count": zones.isna().sum(),
    "missing_pct": zones.isna().mean().mul(100).round(2),
})
print(zone_summary.to_string())

print("\nFirst 5 rows:")
print(zones.head().to_string(index=False))

if "LocationID" in zones.columns:
    print(f"\nMissing LocationID values: {zones['LocationID'].isna().sum():,}")
    print(
        "Number of duplicated non-null LocationIDs: "
        f"{zones['LocationID'].dropna().duplicated().sum():,}"
    )


=== Parquet File Information ===
File size: 61.19 MB
Number of rows: 3,724,889
Row groups: 4

Storage schema:
VendorID: int32
tpep_pickup_datetime: timestamp[us]
tpep_dropoff_datetime: timestamp[us]
passenger_count: int64
trip_distance: double
RatecodeID: int64
store_and_fwd_flag: large_string
PULocationID: int32
DOLocationID: int32
payment_type: int64
fare_amount: double
extra: double
mta_tax: double
tip_amount: double
tolls_amount: double
improvement_surcharge: double
total_amount: double
congestion_surcharge: double
Airport_fee: double
cbd_congestion_fee: double

=== Trip Columns and Missing Values ===
                                dtype  missing_count  missing_pct
VendorID                        int32              0         0.00
tpep_pickup_datetime   datetime64[us]              0         0.00
tpep_dropoff_datetime  datetime64[us]              0         0.00
passenger_count               float64        1088058        29.21
trip_distance                 float64              0    